# Notebook 3. GPTQ

GPTQ (Frantar et al., ICLR 2023) keeps RTN's storage format and lowers the output error of each layer by quantizing one column at a time and adjusting the columns not yet quantized to compensate. This notebook derives the update from the layer-wise reconstruction objective, implements the algorithm as Pare does, including the Cholesky factorization and the column ordering option, and checks the implementation against `pare.schemes.gptq` before running it on GPT-2.

Prerequisites are Notebooks 1 and 2 and basic linear algebra (matrix inverses, Cholesky factorization).

In [ ]:
import sys

import matplotlib.pyplot as plt
import torch

sys.path.insert(0, "..")   # import Pare from this checkout; `pip install pare-quant` also works
torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. The right objective

RTN minimizes the weight error $\lVert W - \hat{W} \rVert^2$. The model never sees $W$ on its own; it computes $Y = X W^T$ for inputs $X \in \mathbb{R}^{n \times I}$. The quantity that reaches the next layer is the output error

$$\mathcal{L}(\hat{W}) = \lVert X W^T - X \hat{W}^T \rVert_F^2 = \sum_{i=1}^{O} (w_i - \hat{w}_i)\, H\, (w_i - \hat{w}_i)^T, \qquad H = \frac{2}{n} X^T X,$$

where $w_i$ is row $i$ of $W$. Every row shares the same matrix $H \in \mathbb{R}^{I \times I}$, which depends only on the inputs. Its diagonal entry $H_{jj}$ is the mean squared activation of input channel $j$, so an error on column $j$ of $W$ costs $H_{jj}$ times as much as the same error on a channel with unit activation. The factor $2/n$ follows Pare's `HessianAccumulator`; it cancels in the update below.

In [ ]:
torch.manual_seed(5)
n_in = 8
X = torch.randn(100, n_in)
X[:, 3] *= 10.0                       # channel 3 has ten times the activation of the others
H = 2 * X.T @ X / len(X)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(H.numpy(), cmap="Blues")
plt.colorbar(im, ax=axes[0])
axes[0].set_title("H = 2 XᵀX / n")
axes[0].set_xlabel("input channel j")
axes[0].set_ylabel("input channel i")
axes[1].bar(range(n_in), H.diag().numpy())
axes[1].set_title("diagonal of H")
axes[1].set_xlabel("input channel")
axes[1].set_xticks(range(n_in))
plt.tight_layout()
plt.show()
print("H diagonal:", [round(v, 1) for v in H.diag().tolist()])

Channel 3 dominates the diagonal by a factor of about 100, so the same rounding error on column 3 of $W$ costs about 100 times more output error than on any other column.

## 2. Compensating one rounding error

Optimal Brain Surgeon (Hassibi and Stork, 1993) gives the exact answer to the question GPTQ asks at every step. If weight $w_q$ in a row is fixed to its rounded value $\hat{w}_q$, the change to the remaining weights that minimizes $\mathcal{L}$ is

$$\delta = -\frac{w_q - \hat{w}_q}{[H^{-1}]_{qq}}\; [H^{-1}]_{q,:}\,,$$

that is, a multiple of row $q$ of the inverse Hessian. Optimal Brain Quantization applies this one weight at a time, picking the weight with the smallest cost each time and recomputing $H^{-1}$ after every removal. That costs $O(I^3)$ per weight, which rules it out for layers with thousands of columns.

GPTQ makes two changes. It quantizes the columns in a fixed order, so all rows of $W$ can be processed together, and it observes that the sequence of inverse Hessians produced by removing columns 1, 2, 3, ... in order is given by the rows of the upper Cholesky factor of $H^{-1}$. With $C = \mathrm{chol}(H^{-1})$ upper triangular, the update for column $k$ becomes

$$e = \frac{w_k - \hat{w}_k}{C_{kk}}, \qquad w_{k+1:} \mathrel{-}= e\, C_{k,\,k+1:}\,,$$

with one factorization for the whole layer and no inverse updates. GPTQ also applies the updates to the columns outside the current block of 128 lazily, as one matrix product per block, which is what makes it fast on a GPU.

## 3. The algorithm

The function below is the loop in `pare.schemes.gptq._gptq_one_layer`, written out with the same steps.

1. Replace dead input channels (zero activation in the calibration data) by an identity entry in $H$ and zero the corresponding weights.
2. Add damping $\lambda I$ with $\lambda = 0.01 \cdot \mathrm{mean}(\mathrm{diag}\, H)$, so that $H$ is positive definite.
3. Optionally reorder the columns by decreasing $H_{jj}$ (`act_order`).
4. Compute $C = \mathrm{chol}(H^{-1})$, upper triangular.
5. For each block of 128 columns, and each column inside it, round the column, compute the scaled error $e$, update the rest of the block, then propagate the block's errors to all later columns at once.
6. Undo the permutation so that the codes are in the original column order.

In [ ]:
def grid(bits):
    return (-128, 127) if bits == 8 else (0, 2**bits - 1)


def scale_zero(W, bits):
    # Asymmetric per-row scale and zero-point (Notebook 1), shape [n_out, 1] each.
    q_min, q_max = grid(bits)
    w_min = W.amin(dim=1, keepdim=True).clamp(max=0.0)
    w_max = W.amax(dim=1, keepdim=True).clamp(min=0.0)
    s = ((w_max - w_min) / (q_max - q_min)).clamp(min=1e-8)
    z = torch.round(q_min - w_min / s).clamp(q_min, q_max)
    return s, z


def gptq_quantize(W, H, bits=4, group_size=None, damp_percent=0.01, act_order=False, block=128):
    # Returns integer codes [n_out, n_in] in the original column order, plus the
    # per-group (scale, zero) used for each column so the caller can dequantize.
    W, H = W.clone().float(), H.clone().float()
    n_out, n_in = W.shape
    q_min, q_max = grid(bits)

    dead = H.diag() == 0                                   # step 1
    H[dead, dead] = 1.0
    W[:, dead] = 0.0
    H.diagonal().add_(damp_percent * H.diag().mean())      # step 2

    perm = None
    if act_order:                                          # step 3
        perm = H.diag().argsort(descending=True)
        W, H = W[:, perm], H[perm][:, perm]

    H_inv = torch.cholesky_inverse(torch.linalg.cholesky(H))
    C = torch.linalg.cholesky(H_inv, upper=True)        # step 4

    Q = torch.zeros_like(W)
    S, Z = torch.zeros_like(W), torch.zeros_like(W)        # scale and zero-point per column
    if group_size is None:
        s, z = scale_zero(W, bits)                         # one scale per row, fixed up front
    for i1 in range(0, n_in, block):                       # step 5
        i2 = min(i1 + block, n_in)
        W1, E1, C1 = W[:, i1:i2].clone(), torch.zeros(n_out, i2 - i1), C[i1:i2, i1:i2]
        for i in range(i2 - i1):
            col = i1 + i
            if group_size is not None and col % group_size == 0:
                s, z = scale_zero(W1[:, i:i + group_size], bits)   # from the updated weights
            w = W1[:, i]
            q = torch.clamp(torch.round(w / s[:, 0] + z[:, 0]), q_min, q_max)
            Q[:, col], S[:, col], Z[:, col] = q, s[:, 0], z[:, 0]
            e = (w - s[:, 0] * (q - z[:, 0])) / C1[i, i]
            W1[:, i + 1:] -= e[:, None] * C1[i, i + 1:][None, :]
            E1[:, i] = e
        W[:, i2:] -= E1 @ C[i1:i2, i2:]                    # lazy update of the later columns

    if perm is not None:                                   # step 6
        inv = torch.argsort(perm)
        Q, S, Z = Q[:, inv], S[:, inv], Z[:, inv]
    return Q, S, Z


def dequant(Q, S, Z):
    return S * (Q - Z)


def rtn_quantize(W, bits=4, group_size=128):
    n_out, n_in = W.shape
    Wg = W.reshape(n_out, -1, group_size)
    s, z = scale_zero(Wg.reshape(-1, group_size), bits)
    s, z = s.reshape(n_out, -1, 1), z.reshape(n_out, -1, 1)
    q = torch.clamp(torch.round(Wg / s + z), *grid(bits))
    return (s * (q - z)).reshape(n_out, n_in)


# Inputs whose channels are correlated, as transformer activations are: 256 channels
# driven by 32 latent factors plus noise, with one salient channel.
n_in, n_out = 256, 16
torch.manual_seed(3)
mixing = torch.randn(32, n_in) / 32**0.5


def make_inputs(n, salient_channel=10, salient_gain=8.0):
    X = torch.randn(n, 32) @ mixing + 0.1 * torch.randn(n, n_in)
    X[:, salient_channel] *= salient_gain
    return X


W = torch.randn(n_out, n_in) * 0.05
X_calib, X_test = make_inputs(4096), make_inputs(1000)
H = 2 * X_calib.T @ X_calib / len(X_calib)
Y = X_test @ W.T


def output_error(W_hat):
    return ((Y - X_test @ W_hat.T).norm() / Y.norm()).item()


W_rtn = rtn_quantize(W, 4, 128)
W_gptq = dequant(*gptq_quantize(W, H, 4, group_size=128))
print(f"RTN  INT4 g=128: relative output error {output_error(W_rtn):.4f}")
print(f"GPTQ INT4 g=128: relative output error {output_error(W_gptq):.4f}")
print(f"weight MAE: RTN {(W_rtn - W).abs().mean():.5f}, GPTQ {(W_gptq - W).abs().mean():.5f}")

GPTQ's weight error is larger than RTN's, because it moves weights away from their original values on purpose, and its output error, the quantity the objective measures, is a fraction of RTN's. The compensation works through the off-diagonal entries of $H$: a rounding error on one column can be offset by columns whose inputs are correlated with it. With independent input channels $H$ is diagonal, $C$ is diagonal, and GPTQ reduces to RTN.

## 4. Why damping

$H = 2 X^T X / n$ is positive semi-definite. When the calibration set has fewer tokens than the layer has input channels, or when channels are collinear, some eigenvalues are zero and the Cholesky factorization fails. Adding $\lambda I$ lifts every eigenvalue by $\lambda$, and setting $\lambda$ to one percent of the mean diagonal keeps it small relative to the typical curvature.

In [ ]:
d = 64
X_small = torch.randn(20, d)             # fewer tokens than channels: rank-deficient H
H_raw = 2 * X_small.T @ X_small / 20
damp = 0.01 * H_raw.diag().mean()
eig_raw = torch.linalg.eigvalsh(H_raw)
eig_damped = torch.linalg.eigvalsh(H_raw + damp * torch.eye(d))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(eig_raw.clamp(min=1e-12).numpy(), "o-", markersize=4, label="H")
ax.semilogy(eig_damped.numpy(), "s-", markersize=4, label=f"H + {damp:.4f} I")
ax.set_xlabel("eigenvalue index")
ax.set_ylabel("eigenvalue")
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

try:
    torch.linalg.cholesky(H_raw)
    print("Cholesky of H succeeded")
except torch.linalg.LinAlgError as err:
    print("Cholesky of H failed:", str(err).split(".")[0])
torch.linalg.cholesky(H_raw + damp * torch.eye(d))
print(f"{(eig_raw < 1e-6).sum().item()} of {d} eigenvalues of H are numerically zero; "
      "Cholesky of the damped matrix succeeds")

## 5. Column order

GPTQ quantizes columns left to right by default. A column's rounding error can only be compensated by the columns that come after it, so a salient column near the end of the row has nowhere to send its error. `act_order=True` sorts the columns by decreasing $H_{jj}$ before the loop, so the most expensive columns are rounded first while many columns remain to absorb their error. The codes are permuted back afterwards, and the scales stay attached to their original groups, so the storage format does not change.

The cell below moves the salient channel of Section 3 to the last column.

In [ ]:
X_calib, X_test = make_inputs(4096, salient_channel=255), make_inputs(1000, salient_channel=255)
H = 2 * X_calib.T @ X_calib / len(X_calib)
Y = X_test @ W.T

for act_order in (False, True):
    W_hat = dequant(*gptq_quantize(W, H, 4, group_size=None, act_order=act_order))
    print(f"act_order={act_order!s:5}: relative output error {output_error(W_hat):.4f}")

In the README, `act_order=True` lowers GPTQ's WikiText-2 perplexity on Llama-3.1-8B from 11.10 to 6.54 and changes it by at most 0.05 on Qwen2.5-7B and OLMo-3-7B. Why the Llama-3 models depend on the order this strongly is an open question in that document.

## 6. How much calibration data

$H$ is estimated from the calibration tokens. With fewer rows than channels it is rank-deficient, the damping term dominates, and the compensation has little to work with. The cell below quantizes the layer of Section 3 with Hessians from 8 to 2048 calibration rows and measures the output error on held-out inputs.

In [ ]:
X_test = make_inputs(1000)
Y = X_test @ W.T

sizes = [8, 32, 128, 512, 2048]
errors = []
for n in sizes:
    X_c = make_inputs(n)
    H_c = 2 * X_c.T @ X_c / n
    W_hat = dequant(*gptq_quantize(W, H_c, 4, group_size=128))
    errors.append(output_error(W_hat))
    print(f"n = {n:4d} rows: relative output error {errors[-1]:.4f}")
print(f"RTN for reference: {output_error(rtn_quantize(W, 4, 128)):.4f}")

plt.figure(figsize=(7, 4))
plt.plot(sizes, errors, "o-")
plt.xscale("log", base=2)
plt.xlabel("calibration rows")
plt.ylabel("relative output error")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

With 8 rows GPTQ gains almost nothing over RTN, and the error falls as $n$ grows until a few hundred rows, after which it flattens. The GPTQ and AWQ papers calibrate on 128 sequences of 2048 tokens, about 260 thousand rows for layers with 4096 input channels, and Pare's benchmarks use the same setting.

## 7. GPTQ in Pare

`pare.schemes.gptq._gptq_one_layer` is the loop of Section 3. The first cell checks that it returns the same codes as `gptq_quantize` for a per-channel layer with and without `act_order`. The second quantizes GPT-2 with GPTQ and compares it with RTN at the same INT4 setting, using 16 calibration sequences of 512 tokens from the WikiText-2 training split.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from pare import QuantConfig, quantize
from pare.calibration.data import load_wikitext2_calibration
from pare.core.dtype import QuantDtype
from pare.core.scale import compute_scale
from pare.eval.perplexity import evaluate_perplexity
from pare.schemes.gptq import _gptq_one_layer

X_calib = make_inputs(4096)
H = 2 * X_calib.T @ X_calib / len(X_calib)

for act_order in (False, True):
    scale, zero = compute_scale(W, QuantDtype.INT4, granularity="per_channel")
    Q_pare = _gptq_one_layer(W.clone(), H.clone(), scale, zero, QuantDtype.INT4,
                             "per_channel", 128, 0.01, act_order)
    Q_nb, _, _ = gptq_quantize(W, H, 4, group_size=None, act_order=act_order)
    assert torch.equal(Q_pare.float(), Q_nb), f"codes differ with act_order={act_order}"
print("gptq_quantize matches pare.schemes.gptq._gptq_one_layer (per channel)")

In [ ]:
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
calib = load_wikitext2_calibration(tokenizer, n_samples=16, seq_len=512)


def fresh_model():
    return AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)


def ppl(model):
    return evaluate_perplexity(
        model, tokenizer, dataset="wikitext2", seq_len=512, n_samples=20, device="cpu"
    )


results = {"FP32": ppl(fresh_model())}
configs = {
    "RTN INT4 g=128": QuantConfig(bits=4, scheme="rtn", granularity="per_group", group_size=128),
    "GPTQ INT4 g=128": QuantConfig(bits=4, scheme="gptq", granularity="per_group", group_size=128),
}
for label, config in configs.items():
    model = fresh_model()
    quantize(model, config, calibration_data=calib, device="cpu")
    results[label] = ppl(model)

for label, value in results.items():
    print(f"{label:>16}: PPL {value:6.2f}  ({value - results['FP32']:+.2f})")

GPT-2 has no `model.model.layers`, so Pare collects every layer's Hessian in one pass over the calibration data; for Llama-style models it processes one block at a time to bound memory (`pare/calibration/layerwise.py`). The printed deltas show how much of RTN's INT4 loss GPTQ recovers on this model. On the 7B models in the README, GPTQ INT4 adds 0.19 to 0.29 perplexity on Llama-2-7B, Qwen2.5-7B and OLMo-3-7B, and 2.61 and 4.86 on Llama-3-8B and Llama-3.1-8B without `act_order`.

## Exercises

1. In Section 3, set `group_size=None` and compare the output error with the per-group result. Then set `damp_percent=0.0` and explain what happens.
2. In Section 5, place the salient channel at column 0 instead of 255. Does `act_order` still help, and why?
3. Section 6 draws calibration inputs from the same distribution as the test inputs. Draw `X_c` with a different salient channel from `X_test` and watch what happens to the error as `n` grows.

## Summary

| Idea | Formula or setting | Role |
|------|--------------------|------|
| Objective | $\sum_i (w_i - \hat{w}_i) H (w_i - \hat{w}_i)^T$ with $H = 2X^TX/n$ | Output error, not weight error |
| Compensation | $w_{k+1:} \mathrel{-}= e\, C_{k,k+1:}$ with $C = \mathrm{chol}(H^{-1})$ | Later columns absorb each rounding error |
| Damping | $\lambda = 0.01 \cdot \mathrm{mean}(\mathrm{diag}\,H)$ | Makes $H$ positive definite |
| Blocks of 128 | lazy update of later columns | One matrix product per block |
| `act_order` | columns sorted by $H_{jj}$ | Expensive columns rounded first |
| Calibration | 128 sequences of 2048 tokens | Pare's benchmark setting |

Notebook 4 covers AWQ, which keeps the weights' rounding as it is and instead rescales the salient input channels before quantizing.